# IMPORTING LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# LOADING DATA

In [2]:
df1= pd.read_csv("Divvy_Trips_2019_Q1.csv")
df2= pd.read_csv("Divvy_Trips_2020_Q1.csv")

# DATA CLEANING AND COMBINING

In [3]:
df1.drop(columns=["trip_id","bikeid"], inplace=True)

In [4]:
df2.drop(columns=["ride_id","rideable_type","start_lat","start_lng","end_lat","end_lng"], inplace=True)

In [5]:
print(df1.shape, df2.shape)

(365069, 9) (426887, 9)


In [6]:
df1=df1.rename(columns={"start_time":"started_at","end_time":"ended_at","from_station_name":"start_station_name","to_station_name":"end_station_name","from_station_id":"start_station_id","to_station_id":"end_station_id","usertype":"member_casual"})

In [7]:
print(df1["member_casual"].value_counts(dropna=False))
print(df2["member_casual"].value_counts(dropna=False))

member_casual
Subscriber    341906
Customer       23163
Name: count, dtype: int64
member_casual
member    378407
casual     48480
Name: count, dtype: int64


In [8]:
df1["member_casual"] = df1["member_casual"].replace({"Subscriber":"member","Customer":"casual"})

In [9]:
df1.head()

,started_at,ended_at,start_station_id,start_station_name,end_station_id,end_station_name,member_casual,ridelength,day_of_week
0,01/01/2019 00:04,01/01/2019 00:11,199,Wabash Ave & Grand Ave,84,Milwaukee Ave & Grand Ave,member,00:06:30,3
1,01/01/2019 00:08,01/01/2019 00:15,44,State St & Randolph St,624,Dearborn St & Van Buren St (*),member,00:07:21,3
2,01/01/2019 00:13,01/01/2019 00:27,15,Racine Ave & 18th St,644,Western Ave & Fillmore St (*),member,00:13:49,3
3,01/01/2019 00:13,01/01/2019 00:43,123,California Ave & Milwaukee Ave,176,Clark St & Elm St,member,00:29:43,3
4,01/01/2019 00:14,01/01/2019 00:20,173,Mies van der Rohe Way & Chicago Ave,35,Streeter Dr & Grand Ave,member,00:06:04,3


In [10]:
df2.head()

,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ridelength,day_of_week
0,21/01/2020 20:06,21/01/2020 20:14,Western Ave & Leland Ave,239,Clark St & Leland Ave,326.0,member,00:07:31,3
1,30/01/2020 14:22,30/01/2020 14:26,Clark St & Montrose Ave,234,Southport Ave & Irving Park Rd,318.0,member,00:03:43,5
2,09/01/2020 19:29,09/01/2020 19:32,Broadway & Belmont Ave,296,Wilton Ave & Belmont Ave,117.0,member,00:02:51,5
3,06/01/2020 16:17,06/01/2020 16:25,Clark St & Randolph St,51,Fairbanks Ct & Grand Ave,24.0,member,00:08:49,2
4,30/01/2020 08:37,30/01/2020 08:42,Clinton St & Lake St,66,Wells St & Hubbard St,212.0,member,00:05:32,5


In [11]:
print(set(df1.columns) == set(df2.columns))

combined = pd.concat([df1, df2], ignore_index=True)
print("Total rows:", len(combined))  

combined.to_csv("combined.csv", index=False)

True
Total rows: 791956


In [12]:
combined.tail()

,started_at,ended_at,start_station_id,start_station_name,end_station_id,end_station_name,member_casual,ridelength,day_of_week
791951,10/03/2020 10:40,10/03/2020 10:40,675,HQ QR,675.0,HQ QR,casual,00:00:01,3
791952,07/03/2020 15:25,07/03/2020 16:14,161,Rush St & Superior St,240.0,Sheridan Rd & Irving Park Rd,member,00:48:08,7
791953,01/03/2020 13:12,01/03/2020 13:38,141,Clark St & Lincoln Ave,210.0,Ashland Ave & Division St,casual,00:25:51,1
791954,07/03/2020 18:02,07/03/2020 18:13,672,Franklin St & Illinois St,264.0,Stetson Ave & South Water St,member,00:10:33,7
791955,08/03/2020 13:03,08/03/2020 13:32,110,Dearborn St & Erie St,85.0,Michigan Ave & Oak St,casual,00:28:30,1


# EXPLORATORY DATA ANALYSIS

- Handling missing values

In [13]:
dict = {}
for i in list(combined.columns):
    dict[i] = combined[i].isnull().sum()

In [14]:
pd.DataFrame(dict,index=["null count"]).transpose()

,null count
started_at,0
ended_at,0
start_station_id,0
start_station_name,0
end_station_id,1
end_station_name,1
member_casual,0
ridelength,0
day_of_week,0


Since the missing values is in end_station_id and end_station_name the rows with the missing values will be dropped since it can't be figured.

In [15]:
combined.dropna(inplace=True)

- Unique Values

In [16]:
dict = {}
for i in list(combined.columns):
    dict[i] = combined[i].value_counts().shape[0]

pd.DataFrame(dict,index=["unique count"]).transpose()

,unique count
started_at,172751
ended_at,172920
start_station_id,612
start_station_name,636
end_station_id,612
end_station_name,636
member_casual,2
ridelength,10041
day_of_week,7


- Descriptive Analysis

In [17]:
combined.describe()

,start_station_id,end_station_id,day_of_week
count,791955.000000,791955.000000,791955.000000
mean,204.399699,204.379354,3.989685
std,158.918352,159.323552,1.811354
min,2.000000,2.000000,1.000000
25%,77.000000,77.000000,3.000000
50%,174.000000,174.000000,4.000000
75%,291.000000,291.000000,5.000000
max,675.000000,675.000000,7.000000


- Converting categorical variables to have the category datatype.
This conversion helps reduce memory usage and improve performance for certain operations.

In [18]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791955 entries, 0 to 791955
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   started_at          791955 non-null  object 
 1   ended_at            791955 non-null  object 
 2   start_station_id    791955 non-null  int64  
 3   start_station_name  791955 non-null  object 
 4   end_station_id      791955 non-null  float64
 5   end_station_name    791955 non-null  object 
 6   member_casual       791955 non-null  object 
 7   ridelength          791955 non-null  object 
 8   day_of_week         791955 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 60.4+ MB


In [19]:
combined.member_casual = combined.member_casual.astype("category")
combined.day_of_week = combined.day_of_week.astype("category")

- Converting ridelength from object data type to a timedelta and started_at and ended_at to datetime datatypes in order to simplify my analysis

In [20]:
combined = combined.drop(columns=["ridelength"])
combined["started_at"] = pd.to_datetime(combined["started_at"], dayfirst=True, errors="coerce")
combined["ended_at"] = pd.to_datetime(combined["ended_at"], dayfirst=True, errors="coerce")

print(combined["started_at"].isna().sum(), combined["ended_at"].isna().sum())

#Checking for negative values in ridelength
combined["ridelength"] = combined["ended_at"] - combined["started_at"]
combined["ridelength_min"] = combined["ridelength"].dt.total_seconds() / 60

#Filtering out negative values
combined = combined[combined["ridelength_min"] >= 0]


0 0


In [21]:
#Checking for negative or odd durations
print("Negative durations:", combined[combined["ridelength_min"] < 0].shape[0])

Negative durations: 0


In [22]:
neg = combined[combined["ridelength_min"] < 0]
print(neg[["started_at", "ended_at", "ridelength_min"]].sort_values("ridelength_min"))

Empty DataFrame
Columns: [started_at, ended_at, ridelength_min]
Index: []


- Since i can't tell what the true duration should be i will drop the rows with negative duration and keep a record for documentation

In [23]:
before = len(combined)

removed_negative = combined[combined["ridelength_min"] < 0]
removed_negative.to_csv("removed_negative_durations.csv", index=False)

combined_clean = combined[combined["ridelength_min"] >= 0].copy()
print(before, "->", len(combined_clean), "rows;", before - len(combined_clean), "removed")

791930 -> 791930 rows; 0 removed


In [24]:
combined_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791930 entries, 0 to 791955
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype          
---  ------              --------------   -----          
 0   started_at          791930 non-null  datetime64[ns] 
 1   ended_at            791930 non-null  datetime64[ns] 
 2   start_station_id    791930 non-null  int64          
 3   start_station_name  791930 non-null  object         
 4   end_station_id      791930 non-null  float64        
 5   end_station_name    791930 non-null  object         
 6   member_casual       791930 non-null  category       
 7   day_of_week         791930 non-null  category       
 8   ridelength          791930 non-null  timedelta64[ns]
 9   ridelength_min      791930 non-null  float64        
dtypes: category(2), datetime64[ns](2), float64(2), int64(1), object(2), timedelta64[ns](1)
memory usage: 55.9+ MB


# DESCRIPTIVE ANALYSIS

1.How many Rides does each group take?

In [25]:
counts = combined_clean["member_casual"].value_counts()
print(counts)

#Count in percentage
print((counts / counts.sum() * 100).round(1))

member_casual
member    720313
casual     71617
Name: count, dtype: int64
member_casual
member    91.0
casual     9.0
Name: count, dtype: float64


- The annual members prove to be the ones with most rides.

2. What is the Ride Length by rider type?

In [26]:
ride_stats = (combined_clean.groupby("member_casual")["ridelength_min"].agg(["count", "mean", "median", "min", "max"]).round(2))
print(ride_stats)

                count   mean  median  min       max
member_casual                                      
casual          71617  84.87    22.0  0.0  177200.0
member         720313  13.25     8.0  0.0  101607.0


- The typical median casual ride lasts 22 minutes which is nearly three times longer than the typical member ride which lasts 8 minutes.
- This can suggest that short, quick trips fit commuting or errands, which is how members tend to use bikes.
- Longer rides fit leisure or sightseeing, which is how casual riders tend to use them.
- From the Divvy website we see that an annual rate is $143.90 per year compared the $19.90 day pass an annual member likely to be using the bike daily for commuting and errands saves with the annual subscription and extra trips will cost them nothing.


3. Which day of the week are most Rides made?

In [27]:
day_names = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}

rides_by_day = (combined_clean.groupby(["member_casual", "day_of_week"])
                .size().unstack(0))
rides_by_day.index = rides_by_day.index.map(day_names)
print(rides_by_day)

member_casual  casual  member
day_of_week                  
Sun             18669   60264
Mon              6736  110441
Tue              8006  127947
Wed              8404  121918
Thu              7809  125213
Fri              8530  115138
Sat             13463   59392


In [28]:
day_names = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}

counts = (combined_clean.groupby(["day_of_week", "member_casual"])
          .size().unstack())

share = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)
share.index = share.index.map(day_names)
print(share)

member_casual  casual  member
day_of_week                  
Sun              23.7    76.3
Mon               5.7    94.3
Tue               5.9    94.1
Wed               6.4    93.6
Thu               5.9    94.1
Fri               6.9    93.1
Sat              18.5    81.5


In [29]:
combined_clean["weekend"] = combined_clean["day_of_week"].isin([1, 7]) 

wk = (combined_clean.groupby(["weekend", "member_casual"]).size().unstack())
wk_share = wk.div(wk.sum(axis=1), axis=0).mul(100).round(1)
wk_share.index = wk_share.index.map({False: "Weekday", True: "Weekend"})
print(wk_share)

member_casual  casual  member
weekend                      
Weekday           6.2    93.8
Weekend          21.2    78.8


- This shows that Casual members ride more on weekends which is shown by higher number of rides on Sunday and Saturday.
- During the weekdays the annual members have significant number of rides.

In [30]:
# Average ride length by day of week
avg_by_day = (combined_clean.groupby(["member_casual", "day_of_week"])["ridelength_min"]
              .mean().unstack(0).round(2))
avg_by_day.index = avg_by_day.index.map(day_names)
print(avg_by_day)

member_casual  casual  member
day_of_week                  
Sun             84.37   16.21
Mon             65.29   13.71
Tue             70.12   12.81
Wed             68.29   11.87
Thu            128.91   11.80
Fri             95.35   13.28
Sat             82.31   16.23


4. What time of the day are most rides made?

In [31]:
combined_clean["hour"] = combined_clean["started_at"].dt.hour
rides_by_hour = combined_clean.groupby(["hour", "member_casual"]).size().unstack()
print(rides_by_hour)

member_casual  casual  member
hour                         
0                 465    2957
1                 331    1743
2                 248    1129
3                 145     896
4                  95    1984
5                 196    9043
6                 482   27701
7                1045   58620
8                2009   78714
9                2636   36004
10               3844   24278
11               4828   29291
12               6044   33557
13               7498   33022
14               7925   32181
15               7996   43018
16               7665   76363
17               7004   95865
18               4272   55562
19               2472   32571
20               1535   19216
21               1298   13621
22                937    8289
23                647    4688


- We see that Annual member peak at 7-8am and 4-5pm which can signify rush hours.
- Casual rides tend to peak midday and afternoon which can signify leisure rides.

5. Which Stations have more members?

In [32]:
for group in ["member", "casual"]:
    print(group)
    print(combined_clean[combined_clean["member_casual"] == group]
          ["start_station_name"].value_counts().head(10))

member
start_station_name
Canal St & Adams St             13799
Clinton St & Washington Blvd    13434
Clinton St & Madison St         12891
Kingsbury St & Kinzie St         8720
Columbus Dr & Randolph St        8515
Canal St & Madison St            7957
Franklin St & Monroe St          7010
Michigan Ave & Washington St     6686
Larrabee St & Kingsbury St       6467
Clinton St & Lake St             6439
Name: count, dtype: int64
casual
start_station_name
HQ QR                           3740
Streeter Dr & Grand Ave         2749
Lake Shore Dr & Monroe St       2732
Shedd Aquarium                  1832
Millennium Park                 1406
Michigan Ave & Oak St           1017
Michigan Ave & Washington St     839
Dusable Harbor                   832
Adler Planetarium                827
Theater on the Lake              795
Name: count, dtype: int64


6. Which group makes more round trips?

In [33]:
combined_clean["round_trip"] = (combined_clean["start_station_id"] ==
                                combined_clean["end_station_id"])
print(combined_clean.groupby("member_casual")["round_trip"].mean().mul(100).round(1))

member_casual
casual    18.8
member     1.9
Name: round_trip, dtype: float64


- A higher round trip in casual rides may suggest leisure use as the members ride and return to the same place.

# Summary Tables

- To check the average time members spend riding the bike

In [34]:
# 1. Median ride length by rider type
(combined_clean.groupby("member_casual")["ridelength_min"].median().reset_index(name="median_ride_min").to_csv("summary_median_ride_length.csv", index=False))


- To compare weekend vs weekday

In [35]:
#2. Rides by day of week
day_names = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
by_day = (combined_clean.groupby(["day_of_week", "member_casual"]).agg(rides=("started_at", "count"),avg_ride_min=("ridelength_min", "mean")).reset_index())
by_day["day_name"] = by_day["day_of_week"].map(day_names)
by_day.to_csv("summary_by_day.csv", index=False)

- To check the bussiest time in a day

In [36]:
#3 Rides by hour
(combined_clean.groupby(["hour", "member_casual"]).size().reset_index(name="rides").to_csv("summary_by_hour.csv", index=False))

- To check which members do more round trips

In [37]:
# 4. Round trips
(combined_clean.groupby("member_casual")["round_trip"].mean().mul(100).round(1).reset_index(name="round_trip_pct").to_csv("summary_round_trips.csv", index=False))

- To check top stations

In [38]:
# 5. Top 10 start stations per group
top_stations = (combined_clean.groupby(["member_casual", "start_station_name"]).size().reset_index(name="rides").sort_values(["member_casual", "rides"], ascending=[True, False]).groupby("member_casual").head(10))
top_stations.to_csv("summary_top_stations.csv", index=False)

- All the summary tables are saved in a csv file and visualizations will be done on Tableau 

Tableau link: https://public.tableau.com/views/Cyclistic_Case_Study_17937121519820/Dashboard?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link